# 00 · Local copy of the Hugging Face models (optional)

**Why:** the training notebooks download their backbone from the Hugging Face Hub, which needs **Internet on**. For a GPU session without Internet, this notebook downloads the models once (no GPU needed) and its output is attached instead: `src.cross_encoder.model_source` finds `hf_models/<org>__<model>/` in the inputs and loads from there; the run files keep the Hub name in `cfg['model']`.

| Model | Used by |
|---|---|
| `cross-encoder/ms-marco-MiniLM-L-12-v2` | 03, 13, 14 |
| `microsoft/deberta-v3-base` | 15 |
| `microsoft/deberta-v3-large` | 18 (timing only: inference cost does not depend on the fine-tuned weights) |

**Kaggle settings:** Accelerator **None** (CPU), **Internet on**. A few minutes. Required for any GPU session without Internet: attach its output to 03 (upload copy), 13, 14 and 15.

**Output** (`/kaggle/working/hf_models/`): one folder per model (config, tokenizer, weights). Attach it with Add Input → Notebook Output → **00_hf_models**.

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer

MODELS = ['cross-encoder/ms-marco-MiniLM-L-12-v2', 'microsoft/deberta-v3-base', 'microsoft/deberta-v3-large']
IGNORE = ['onnx/*', 'openvino/*', '*.onnx', '*.h5', '*.msgpack', '*.ot', 'tf_model*', 'flax_model*', 'rust_model*']
OUT = Path('/kaggle/working/hf_models')

for name in MODELS:
    d = OUT / name.replace('/', '__')
    snapshot_download(name, local_dir=d, ignore_patterns=IGNORE)
    AutoConfig.from_pretrained(d)
    AutoTokenizer.from_pretrained(d).save_pretrained(d)   # loads without the Hub; saves the fast tokenizer.json too
    size = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 2**20
    print(f'{name}: {d} | {size:.0f} MB | {sorted(f.name for f in d.iterdir() if f.is_file())}')